# 🤖 03 - Modeling & Evaluation

Train and compare:
- Logistic Regression (baseline)
- Random Forest
- XGBoost
- LightGBM
- Isolation Forest (unsupervised)

**Evaluation metrics:**
- PR-AUC (primary — not ROC-AUC which is misleading on imbalanced data)
- Business cost function (net dollars saved)
- Precision/Recall tradeoff

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from src.train import FraudTrainer, IsolationForestDetector, get_model_descriptions
from src.evaluate import FraudEvaluator, print_evaluation_summary

# Load preprocessed data
X_train = joblib.load('data/processed/X_train.pkl')
X_test = joblib.load('data/processed/X_test.pkl')
y_train = joblib.load('data/processed/y_train.pkl')
y_test = joblib.load('data/processed/y_test.pkl')

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train fraud: {y_train.sum()}, Test fraud: {y_test.sum()}')

## 1. Model Descriptions

In [ ]:
for name, desc in get_model_descriptions().items():
    print(f'\n{name}:')
    print(f'  {desc}')

## 2. Train All Models

In [ ]:
# Train supervised models
trainer = FraudTrainer()
models = trainer.train_all(X_train, y_train)

# Train Isolation Forest (unsupervised)
iso_detector = IsolationForestDetector(contamination=0.005)
iso_detector.fit(X_train, y_train)

print(f'\nTrained {len(models)} supervised models + Isolation Forest')

## 3. Evaluate All Models

In [ ]:
# Get predictions from all models
evaluator = FraudEvaluator(avg_fraud_loss=150, review_cost=5)

predictions = {}
results = {}

# Supervised models
for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    predictions[name] = y_proba
    results[name] = evaluator.evaluate_model(y_test, y_proba, model_name=name)
    print(print_evaluation_summary(results[name]))

# Isolation Forest
iso_probas = iso_detector.predict_proba_as_fraud(X_test)
predictions['isolation_forest'] = iso_probas
results['isolation_forest'] = evaluator.evaluate_model(y_test, iso_probas, model_name='isolation_forest')
print(print_evaluation_summary(results['isolation_forest']))

## 4. Model Comparison Table

In [ ]:
comparison = evaluator.compare_models(y_test, predictions)
print('\n=== MODEL COMPARISON ===')
print(comparison.to_string(index=False))

## 5. Precision-Recall Curves & Cost Analysis

In [ ]:
# PR Curves
fig = evaluator.plot_precision_recall_curve(y_test, predictions, save_path='data/processed/pr_curves.png')
plt.show()

In [ ]:
# Cost vs Threshold for best model
best_model_name = comparison.iloc[0]['Model']
fig = evaluator.plot_cost_vs_threshold(y_test, predictions[best_model_name],
                                        model_name=best_model_name,
                                        save_path='data/processed/cost_vs_threshold.png')
plt.show()
print(f'\nBest model: {best_model_name}')
print(f'This plot shows why default threshold of 0.5 is often suboptimal.')

## 6. Save Best Model

In [ ]:
# Save the best model
best_result = results[best_model_name]
trainer.save_model(best_model_name, f'models/{best_model_name}.pkl')

# Save threshold
with open('models/threshold.txt', 'w') as f:
    f.write(str(best_result['threshold']))

# Save comparison results
comparison.to_csv('data/processed/model_comparison.csv', index=False)

print(f'\nSaved best model: {best_model_name}')
print(f'Optimal threshold: {best_result["threshold"]}')
print(f'Net benefit: ${best_result["business"]["net_benefit_usd"]:,.2f}')
print('\n✅ Modeling complete. Ready for explainability.')